In [1]:
import pandas as pd
import numpy as np
import requests
import time
from tqdm import tqdm
from tmdbv3api import TMDb, Movie

In [2]:
# ==========================================
# 1. SETUP AND AUTHENTICATION
# ==========================================
tmdb = TMDb()
tmdb.api_key = 'aa449cf1f3342eb70dc0e485d2372de6'  # <--- MUST REPLACE THIS!
tmdb_movie = Movie()

# Enable the progress bar for Pandas
tqdm.pandas(desc="Fetching TMDB Data")

# Our language mapping dictionary from earlier
language_mapping = {
    'bg': 'Bulgarian', 'ca': 'Catalan', 'da': 'Danish', 'de': 'German',
    'el': 'Greek', 'en': 'English', 'es': 'Spanish', 'fa': 'Persian',
    'fi': 'Finnish', 'fr': 'French', 'he': 'Hebrew', 'hi': 'Hindi',
    'hu': 'Hungarian', 'it': 'Italian', 'ja': 'Japanese', 'ko': 'Korean',
    'nl': 'Dutch', 'pl': 'Polish', 'pt': 'Portuguese', 'ru': 'Russian',
    'sr': 'Serbian', 'sv': 'Swedish', 'ta': 'Tamil', 'te': 'Telugu',
    'tr': 'Turkish', 'uk': 'Ukrainian', 'zh': 'Chinese'
}


In [3]:
# ==========================================
# 2. SCRAPING WIKIPEDIA (2019 - 2025)
# ==========================================
print("Phase 1: Scraping Wikipedia...")
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
all_years_data = []

for year in range(2019, 2026):
    print(f" -> Pulling movies for {year}...")
    link = f'https://en.wikipedia.org/wiki/List_of_American_films_of_{year}'
    
    try:
        # We use match='Title' to ONLY grab tables that contain movie data, 
        # ignoring random summary tables on the page!
        tables = pd.read_html(link, match='Title', storage_options=headers)
        
        # Combine all valid movie tables for this year
        yearly_df = pd.concat(tables, ignore_index=True)
        yearly_df['Release_Year'] = year
        all_years_data.append(yearly_df)
        
        # Pause to avoid getting blocked by Wikipedia
        time.sleep(2)
        
    except Exception as e:
        print(f" -> [Warning] Could not scrape {year}: {e}")

# Combine all 8 years into one master DataFrame
master_df = pd.concat(all_years_data, ignore_index=True)

# Clean up the 'Title' column and drop any empty rows
master_df.rename(columns={'Title': 'movie_title'}, inplace=True)
master_df.dropna(subset=['movie_title'], inplace=True)
# Keep only the columns we care about so far to save memory
master_df = master_df[['movie_title', 'Release_Year']].copy()

print(f"\nSuccessfully scraped {len(master_df)} movies from Wikipedia!")

Phase 1: Scraping Wikipedia...
 -> Pulling movies for 2019...
 -> Pulling movies for 2020...
 -> Pulling movies for 2021...
 -> Pulling movies for 2022...
 -> Pulling movies for 2023...
 -> Pulling movies for 2024...
 -> Pulling movies for 2025...

Successfully scraped 2622 movies from Wikipedia!


In [4]:
# ==========================================
# 3. TMDB API FETCH FUNCTION
# ==========================================
def get_full_movie_data(title):
    # Pause for 0.25 seconds to respect TMDB's rate limits safely
    time.sleep(0.25)
    
    empty_result = pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan])
    
    try:
        search_result = tmdb_movie.search(title)
        if not search_result:
            return empty_result
            
        movie_id = search_result[0].id
        
        # Fetch details AND cast/crew using append_to_response to save API calls
        url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}&append_to_response=credits'
        response = requests.get(url)
        data = response.json()
        
        # Extract Basic Details
        language = data.get('original_language', np.nan)
        duration = data.get('runtime', np.nan)
        tmdb_score = data.get('vote_average', np.nan)
        
        # Extract Genres
        genres = np.nan
        if data.get('genres'):
            genres = ", ".join([g['name'] for g in data['genres']])
            
        # Extract Director and Actors
        director_name = np.nan
        actor_1, actor_2, actor_3 = np.nan, np.nan, np.nan
        
        if data.get('credits'):
            for member in data['credits'].get('crew', []):
                if member.get('job') == 'Director':
                    director_name = member.get('name')
                    break 
            
            cast = data['credits'].get('cast', [])
            if len(cast) > 0: actor_1 = cast[0].get('name')
            if len(cast) > 1: actor_2 = cast[1].get('name')
            if len(cast) > 2: actor_3 = cast[2].get('name')
            
        return pd.Series([director_name, actor_1, actor_2, actor_3, genres, duration, language, tmdb_score])
        
    except Exception:
        return empty_result

In [5]:
# ==========================================
# 4. APPLYING TMDB DATA & FINAL CLEANUP
# ==========================================
print("\nPhase 2: Fetching TMDB data (This will take a while, grab a coffee!)")

columns_to_fill = ['director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name', 'genres', 'duration', 'language', 'tmdb_score']

# Run the API fetch across every movie
master_df[columns_to_fill] = master_df['movie_title'].progress_apply(lambda x: get_full_movie_data(str(x)))

print("\nPhase 3: Final Data Cleaning...")

# Round numerical columns to 1 decimal place
master_df[['duration', 'tmdb_score']] = master_df[['duration', 'tmdb_score']].round(1)

# Convert the language abbreviations into full names
# (If an abbreviation isn't in our dictionary, we'll just keep the original text)
master_df['language'] = master_df['language'].replace(language_mapping)

# Reorder columns to look clean and professional
final_columns = [
    'movie_title', 'Release_Year', 'director_name', 'actor_1_name', 
    'actor_2_name', 'actor_3_name', 'genres', 'duration', 'language', 'tmdb_score'
]
master_df = master_df[final_columns]


Phase 2: Fetching TMDB data (This will take a while, grab a coffee!)


Fetching TMDB Data: 100%|██████████| 2622/2622 [1:16:04<00:00,  1.74s/it]


Phase 3: Final Data Cleaning...


In [8]:
# ==========================================
# 5. SAVE THE FINAL DATASET
# ==========================================
output_filename = 'American_Movies_2019_to_2025_Master.csv'
master_df.to_csv(output_filename, index=False)

print(f"\nAll Done! Saved perfectly clean dataset to {output_filename}.")
print(master_df.tail())


All Done! Saved perfectly clean dataset to American_Movies_2019_to_2025_Master.csv.
                   movie_title  Release_Year     director_name  \
2623            Song Sung Blue          2025      Craig Brewer   
2624             Marty Supreme          2025       Josh Safdie   
2625  The Testament of Ann Lee          2025     Mona Fastvold   
2626               Continuance          2025        Tony Olmos   
2627          I Was a Stranger          2025  Reiko Aylesworth   

             actor_1_name         actor_2_name          actor_3_name  \
2623         Hugh Jackman          Kate Hudson     Michael Imperioli   
2624    Timothée Chalamet      Gwyneth Paltrow         Odessa A'zion   
2625      Amanda Seyfried        Lewis Pullman     Thomasin McKenzie   
2626      Tony Gorodeckas         Noor Razooky  Teresa Suarez Grosso   
2627  Elizabeth Rodriguez  Jason Butler Harner        Carlease Burke   

                     genres  duration language  tmdb_score  
2623  Music, Romance, Dr